In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

def run_z_test(file_name, skill_name, year_a, year_b):
    path = os.path.join(BASE_DIR, "data", "processed", file_name)
    df = pd.read_csv(path)
    
    skill_col = 'hard_skills' if 'HARD' in file_name else 'soft_skills'
    
    data_a = df[(df['year'] == year_a) & (df[skill_col] == skill_name)]
    data_b = df[(df['year'] == year_b) & (df[skill_col] == skill_name)]
    
    count_a = data_a['skill_count'].iloc[0] if not data_a.empty else 0
    nobs_a = df[df['year'] == year_a]['total_jobs'].iloc[0]
    
    count_b = data_b['skill_count'].iloc[0] if not data_b.empty else 0
    nobs_b = df[df['year'] == year_b]['total_jobs'].iloc[0]
    
    stat, pval = proportions_ztest([count_a, count_b], [nobs_a, nobs_b])
    
    print(f"\nTEST: {skill_name} ({year_a} vs {year_b})")
    print(f"P1: {count_a}/{nobs_a} ({round(count_a/nobs_a*100, 2)}%)")
    print(f"P2: {count_b}/{nobs_b} ({round(count_b/nobs_b*100, 2)}%)")
    print(f"P-VALUE: {pval:.4e}")
    print("SIGNIFICANT" if pval < 0.05 else "NOT SIGNIFICANT")

run_z_test("IT_TOP_30_GLOBAL_HARD.csv", "Artificial Intelligence", 2021, 2024)
run_z_test("IT_TOP_30_GLOBAL_SOFT.csv", "Innovation", 2018, 2024)

In [ ]:
import pandas as pd
import ast
import os
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "IT_LIGHTCAST_SKILLS.csv")

def run_clustering(n_clusters=4):
    df = pd.read_csv(INPUT_FILE, usecols=['hard_skills'], low_memory=False).dropna().head(20000)
    df['skills_str'] = df['hard_skills'].apply(lambda x: " ".join(ast.literal_eval(x)))
    
    vectorizer = CountVectorizer(max_features=500)
    X = vectorizer.fit_transform(df['skills_str'])
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_transform(X).argmin(axis=1)
    
    feature_names = vectorizer.get_feature_names_out()
    
    print(f"\n--- SKILL CLUSTERS (K={n_clusters}) ---")
    for i in range(n_clusters):
        center = kmeans.cluster_centers_[i]
        top_indices = center.argsort()[-10:][::-1]
        top_skills = [feature_names[idx] for idx in top_indices]
        print(f"Cluster {i}: {', '.join(top_skills)}")

run_clustering(n_clusters=5)

In [ ]:
def check_skill_trend_single_source(domain, target_skill, source_name, skill_type='hard'):
    df_path = os.path.join(BASE_DIR, "data", "processed", 
                           f"{domain}_LIGHTCAST_SKILLS.csv")
    df = pd.read_csv(df_path, usecols=['year_month', 'source', f'{skill_type}_skills'])
    df['year'] = df['year_month'].astype(str).str[:4].astype(int)
    
    df = df[df['source'] == source_name].copy()
    
    year_counts = df.groupby('year').size()
    valid_years = year_counts[year_counts >= 100].index
    df = df[df['year'].isin(valid_years)]
    
    if df['year'].nunique() < 3:
        print(f"Not enough years with sufficient data for {target_skill} in {source_name}")
        return
    
    df['has_skill'] = df[f'{skill_type}_skills'].apply(
        lambda x: 1 if target_skill in str(x) else 0
    )
    
    try:
        model = smf.logit('has_skill ~ year', data=df).fit(disp=0)
        coef = model.params['year']
        pval = model.pvalues['year']
        odds_ratio = pd.np.exp(coef) if hasattr(pd, 'np') else __import__('numpy').exp(coef)
        
        print(f"\n--- {target_skill} | {domain} | {source_name} ---")
        print(f"Years analysed: {sorted(df['year'].unique().tolist())}")
        print(f"N observations: {len(df):,}")
        print(f"Coefficient:    {coef:.4f}")
        print(f"Odds Ratio:     {odds_ratio:.4f}  (per additional year)")
        print(f"P-Value:        {pval:.4e}")
        
        if pval < 0.05:
            direction = "INCREASING" if coef > 0 else "DECREASING"
            pct_change = (odds_ratio - 1) * 100
            print(f"Conclusion: Significant {direction} trend.")
            print(f"Interpretation: Each year, odds of seeing this skill "
                  f"{'increase' if coef > 0 else 'decrease'} "
                  f"by {abs(pct_change):.1f}%")
        else:
            print("Conclusion: No significant trend.")
    except Exception as e:
        print(f"Model error: {e}")

check_skill_trend_single_source("IT", "Linux", "stackoverflow")
check_skill_trend_single_source("IT", "Docker", "stackoverflow")
check_skill_trend_single_source("IT", "Machine Learning", "stackoverflow")
check_skill_trend_single_source("Finance", "Computer Science", "efinancialcareers")
check_skill_trend_single_source("Finance", "Risk Management", "efinancialcareers")